# Importations nécessaire

In [1]:
import numpy as np
from gurobipy import *
import pandas as pd

# Préparation des données
Récuperation des matrices de gains, de coût, des listes des bâtiments et de leurs types

In [2]:
# --- 1. Lecture du fichier ---
df = pd.read_excel("efficacity_data_v1.xlsx")

# ============================
# PARTIE A — CALIBRATED
# ============================

df_calibrated = (
    df[df["id_simulation"].astype(str).str.contains("calibrated", case=False)]
    [["building_name", "building_type", "id_simulation", "conso_total_mwh_an"]]
)

# Comptage par catégorie
nb_buildings_by_type = (
    df_calibrated
    .groupby("building_type")["building_name"]
    .count()
    .to_dict()
)

# Liste des calibrated par bâtiment
calibrated_list = (
    df_calibrated
    .groupby("building_name")["conso_total_mwh_an"]
    .apply(list)
    .to_dict()
)

# ============================
# PARTIE B — SANS CALIBRATED
# ============================

df_nc = df[
    ~df["id_simulation"].astype(str).str.contains("calibrated", case=False)
]

# Colonnes utiles
df_nc = df_nc[[
    "building_name",
    "id_simulation",
    "building_type",
    "gains_totaux_mwh_an",
    "cout_investissement_euros"
]]

# --- 2. Groupement ---
group_gains = df_nc.groupby("building_name")["gains_totaux_mwh_an"].apply(list)
group_cost  = df_nc.groupby("building_name")["cout_investissement_euros"].apply(list)

# --- 3. Taille maximale ---
max_len = max(
    group_gains.apply(len).max(),
    group_cost.apply(len).max()
)

# --- 4. Matrices à taille fixe ---
gains_matrix = np.array([
    vals + [np.inf] * (max_len - len(vals))
    for vals in group_gains
])

cost_matrix = np.array([
    vals + [np.inf] * (max_len - len(vals))
    for vals in group_cost
])

group_type = (
    df_nc
    .groupby("building_name")["building_type"]
    .first()
)
# --- 5. Métadonnées utiles ---
building_names = group_gains.index.tolist()
building_types = group_type.loc[building_names].tolist()

grouped = df_nc.groupby("building_name", sort=False)["id_simulation"].apply(list)
renovation_names_per_building = grouped.to_dict()

print("gains matrix shape :", gains_matrix.shape)
print("Cost matrix shape  :", cost_matrix.shape)
print("Nb bâtiments       :", len(building_names))
print("Nb calibrated      :", len(df_calibrated))
print(df_calibrated.head())
print(nb_buildings_by_type)

gains matrix shape : (71, 15)
Cost matrix shape  : (71, 15)
Nb bâtiments       : 71
Nb calibrated      : 76
                             building_name   building_type  \
0                            abri_bouliste          Sports   
16  accueil_de_loisirs_maternel_la_varenne    Enseignement   
32             association_31_rue_gambetta          Autres   
48               ateliers_municipaux_(ctm)  Administration   
59                     base_de_canoe_kayak          Sports   

                                        id_simulation  conso_total_mwh_an  
0                        abri_bouliste_ref_calibrated           13.256400  
16  accueil_de_loisirs_maternel_la_varenne_ref_cal...           22.254286  
32         association_31_rue_gambetta_ref_calibrated           12.827150  
48           ateliers_municipaux_(ctm)_ref_calibrated         1148.106350  
59                 base_de_canoe_kayak_ref_calibrated           21.643910  
{'Administration': 6, 'Autres': 26, 'Enseignement': 29, 'Sports

(Optionnel) Sauvegarde de ces matrices sous format csv

In [30]:
""" # Sauvegarde des matrices
np.savetxt(
    "gains_matrix.csv",
    gains_matrix,
    delimiter=",",
    fmt="%.10g"
)

np.savetxt(
    "cost_matrix.csv",
    cost_matrix,
    delimiter=",",
    fmt="%.10g"
)

np.savetxt(
    "calibrated_list.csv",
    np.array(list(calibrated_list.keys()), dtype=str),
    delimiter=",",
    fmt="%s"
)

# Sauvegarde des bâtiments (utile pour l’indexation)
np.savetxt(
    "building_names.csv",
    np.array(building_names, dtype=str),
    delimiter=",",
    fmt="%s"
)
np.savetxt(
    "building_types.csv",
    np.array(building_types, dtype=str),
    delimiter=",",
    fmt="%s"
) """

' # Sauvegarde des matrices\nnp.savetxt(\n    "gains_matrix.csv",\n    gains_matrix,\n    delimiter=",",\n    fmt="%.10g"\n)\n\nnp.savetxt(\n    "cost_matrix.csv",\n    cost_matrix,\n    delimiter=",",\n    fmt="%.10g"\n)\n\nnp.savetxt(\n    "calibrated_list.csv",\n    np.array(list(calibrated_list.keys()), dtype=str),\n    delimiter=",",\n    fmt="%s"\n)\n\n# Sauvegarde des bâtiments (utile pour l’indexation)\nnp.savetxt(\n    "building_names.csv",\n    np.array(building_names, dtype=str),\n    delimiter=",",\n    fmt="%s"\n)\nnp.savetxt(\n    "building_types.csv",\n    np.array(building_types, dtype=str),\n    delimiter=",",\n    fmt="%s"\n) '

Chargement des matrices (si sauvegardées)

In [31]:
""" # Chargement des matrices
gains_matrix = np.loadtxt(
    "gains_matrix.csv",
    delimiter=","
)

cost_matrix = np.loadtxt(
    "cost_matrix.csv",
    delimiter=","
)

# Chargement des noms de bâtiments
building_names = np.loadtxt(
    "building_names.csv",
    delimiter=",",
    dtype=str
).tolist()

building_types = np.loadtxt(
    "building_types.csv",
    delimiter=",",
    dtype=str
).tolist()

print("gains matrix shape :", gains_matrix.shape)
print("Cost matrix shape  :", cost_matrix.shape)
print("Nb bâtiments       :", len(building_names))
print("Nb calibrated      :", len(df_calibrated))
print(df_calibrated.head()) """

' # Chargement des matrices\ngains_matrix = np.loadtxt(\n    "gains_matrix.csv",\n    delimiter=","\n)\n\ncost_matrix = np.loadtxt(\n    "cost_matrix.csv",\n    delimiter=","\n)\n\n# Chargement des noms de bâtiments\nbuilding_names = np.loadtxt(\n    "building_names.csv",\n    delimiter=",",\n    dtype=str\n).tolist()\n\nbuilding_types = np.loadtxt(\n    "building_types.csv",\n    delimiter=",",\n    dtype=str\n).tolist()\n\nprint("gains matrix shape :", gains_matrix.shape)\nprint("Cost matrix shape  :", cost_matrix.shape)\nprint("Nb bâtiments       :", len(building_names))\nprint("Nb calibrated      :", len(df_calibrated))\nprint(df_calibrated.head()) '

Remarque: Il y a 76 lignes avec calibrated, dont il ne reste que 71 après sélection des scénarii de simulation, il y a donc 5 bâtiments sur lesquels aucune rénovation n'est possible, mais qui entrent quand même en jeu dans la consommation totale

In [3]:
Conso_total = np.sum(df_calibrated["conso_total_mwh_an"])
nbr_annees = 2050 - 2026 + 1
nbr_batiments = gains_matrix.shape[0]
nbr_renovations_max = gains_matrix.shape[1]

Les variables de décisions sont:
$$ x_{b,r,t} = 1 \quad \text{Si la rénovation r est réalisée sur le bâtiment b à l'année t, 0 sinon}

In [4]:
m = Model("Efficacity Optimization")
X = m.addMVar(shape = (nbr_batiments, nbr_renovations_max, nbr_annees), vtype=GRB.BINARY, name="X")


Set parameter Username
Set parameter LicenseID to value 2773884
Academic license - for non-commercial use only - expires 2027-02-02


Contrainte: 1 rénovation par bâtiment sur la période
$$ \forall b \quad \sum_{r,t}x_{b,r,t} \leq 1

In [5]:
for b in range(nbr_batiments):
    m.addConstr(
        quicksum(X[b, r, t] for r in range(nbr_renovations_max) for t in range(nbr_annees)) <= 1,
        name=f"one_renovation_per_building_{b}"
    )

Alternative si on s'autorise plus d'une rénovation:
$$ \forall b \quad \forall r \sum_t x_{b,r,t} \leq 1

In [ ]:
""" for b in range(nbr_batiments):
    for r in range(nbr_renovations_max):
            m.addConstr(
                quicksum(X[b, r, t] for t in range(nbr_annees)) <= 1,
                name=f"one_renovation_{r}_per_building_{b}"
            ) """

Contrainte de budget annuel:
$$ \forall t \quad \sum_{b,r} x_{b,r,t}*cout_{b,r} \leq 2M€ 

In [6]:
for t in range(nbr_annees):
    m.addConstr(
        quicksum(
            X[b, r, t] * cost_matrix[b, r]
            for b in range(nbr_batiments)
            for r in range(nbr_renovations_max)
            if cost_matrix[b, r] < float("inf")   # ignore rénovations inexistantes
        )
        <= 2e6,
        name=f"budget_year_{t}"
    )

Contrainte: Disponibilité par catégorie
$$ \forall cat \quad \forall t \quad \sum_{b \in cat} \sum_r x_{b,r,t}*\frac{1}{card(cat)} \leq 0.2

In [7]:
from collections import defaultdict

buildings_by_cat = defaultdict(list)

for b, cat in enumerate(building_types):
    buildings_by_cat[cat].append(b)
print(buildings_by_cat)

for cat, b_indices in buildings_by_cat.items():

    card_cat = nb_buildings_by_type[cat]   # card(cat)

    for t in range(nbr_annees):
        m.addConstr(
            quicksum(
                X[b, r, t]
                for b in b_indices
                for r in range(nbr_renovations_max)
            )
            <= 0.2 * card_cat,
            name=f"availability_{cat}_year_{t}"
        )

defaultdict(<class 'list'>, {'Sports': [0, 4, 8, 41, 42, 43, 44, 45, 46, 47, 64, 66, 67, 68, 69], 'Enseignement': [1, 14, 15, 16, 17, 18, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 62], 'Autres': [2, 5, 6, 7, 10, 11, 12, 13, 20, 21, 48, 49, 50, 52, 53, 55, 56, 57, 58, 59, 60, 61, 63, 65, 70], 'Administration': [3, 9, 19, 51, 54]})


Contrainte décret:
$$ \sum_{t=0}^{t=4}\sum_{b,r,t} x_{b,r,2026+t}*gain_{b,r} \geq 0.4*conso_0 \\
   \sum_{t=0}^{t=14}\sum_{b,r,t} x_{b,r,2026+t}*gain_{b,r} \geq 0.5*conso_0 \\
   \sum_{t=0}^{t=24}\sum_{b,r,t} x_{b,r,2026+t}*gain_{b,r} \geq 0.6*conso_0

In [ ]:
""" # Année de départ
t0 = 0  # correspond à 2026

horizons = {
    "40pct_5ans":  (0, 4,  0.4),
   "50pct_15ans": (0, 14, 0.5),
    "60pct_25ans": (0, 24, 0.6)
}

for name, (t_start, t_end, ratio) in horizons.items():

    m.addConstr(
        quicksum(
            X[b, r, t] * gains_matrix[b, r]
            for t in range(t_start, t_end + 1)
            for b in range(nbr_batiments)
            for r in range(nbr_renovations_max  )
            if gains_matrix[b, r] < float("inf")
        )
        >= ratio * Conso_total,
        name=f"decret_{name}"
    )
 """

Fonction objectif:
$$ min(\sum_{b,r,t} x_{b,r,t}*cout_{b,r})

In [ ]:
""" m.setObjective(
    quicksum(
        X[b, r, t] * cost_matrix[b, r]
        for b in range(nbr_batiments)
        for r in range(nbr_renovations_max)
        for t in range(nbr_annees)
        if gains_matrix[b, r] < float("inf")
    ),
    GRB.MINIMIZE
) """

Fonction Objectif alternative: minimiser les écarts par rapport aux objectifs
$$ min((0.4*Conso_0 - \sum_{t=0}^{t=4}\sum_{b,r}x_{b,r,t}*gain_{b,r}) + \\
(0.5*Conso_0 - \sum_{t=0}^{t=14}\sum_{b,r}x_{b,r,t}*gain_{b,r}) + \\
(0.6*Conso_0 - \sum_{t=0}^{t=24}\sum_{b,r}x_{b,r,t}*gain_{b,r}))

In [10]:
# Année de départ
t0 = 0  # correspond à 2026

horizons = {
    "40pct_5ans":  (0, 4,  0.4),
   "50pct_15ans": (0, 14, 0.5),
    "60pct_25ans": (0, 24, 0.6)
}

m.setObjective((0.4*Conso_total-quicksum(X[b,r,t]*gains_matrix[b,r] for t in range(t0, t0+4) for b in range(nbr_batiments) for r in range(nbr_renovations_max) if gains_matrix[b,r] < float("inf")))
               + (0.5*Conso_total-quicksum(X[b,r,t]*gains_matrix[b,r] for t in range(t0, t0+14) for b in range(nbr_batiments) for r in range(nbr_renovations_max) if gains_matrix[b,r] < float("inf")))
               + (0.6*Conso_total-quicksum(X[b,r,t]*gains_matrix[b,r] for t in range(t0, t0+24) for b in range(nbr_batiments) for r in range(nbr_renovations_max) if gains_matrix[b,r] < float("inf")))
               , GRB.MINIMIZE)

Résolution du modèle

In [11]:
m.update()
m.optimize()


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i5-1135G7 @ 2.40GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 196 rows, 26625 columns and 72875 nonzeros (Min)
Model fingerprint: 0x240fc8c3
Model has 18840 linear objective coefficients and an objective constant of 3.6894873122849342e+04
Variable types: 0 continuous, 26625 integer (26625 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+06]
  Objective range  [2e-04, 2e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+06]

Found heuristic solution: objective 25690.909264
Presolve removed 5 rows and 16689 columns
Presolve time: 0.10s
Presolved: 191 rows, 9936 columns, 29808 nonzeros
Found heuristic solution: objective 26878.209951
Variable types: 0 continuous, 9936 integer (9936 binary)
Found heuristic solution: objective 24134.082902

Root

Le modèle ainis défini est infaisable

In [12]:
print(m.ObjVal)

16030.065840993273


In [13]:
obj_2030 = quicksum(
    X[b, r, t] * gains_matrix[b, r]
    for t in range(0, 4)  # 2026-2030
    for b in range(nbr_batiments)
    for r in range(nbr_renovations_max)
    if gains_matrix[b, r] < float("inf")
).getValue()/ Conso_total

obj_2040 = quicksum(
    X[b, r, t] * gains_matrix[b, r]
    for t in range(0, 14)  # 2026-2040
    for b in range(nbr_batiments)
    for r in range(nbr_renovations_max)
    if gains_matrix[b, r] < float("inf")
).getValue()/ Conso_total

obj_2050 = quicksum(
    X[b, r, t] * gains_matrix[b, r]
    for t in range(0, 24)  # 2026-2050
    for b in range(nbr_batiments)
    for r in range(nbr_renovations_max)
    if gains_matrix[b, r] < float("inf")
).getValue()/ Conso_total

print(f"Objectif 2030 : {obj_2030:.4f} (cible 0.4)")
print(f"Objectif 2040 : {obj_2040:.4f} (cible 0.5)")
print(f"Objectif 2050 : {obj_2050:.4f} (cible 0.6)")

Objectif 2030 : 0.1740 (cible 0.4)
Objectif 2040 : 0.3345 (cible 0.5)
Objectif 2050 : 0.3398 (cible 0.6)


In [ ]:
# Création d'une liste de dictionnaires pour construire un DataFrame
results = []

for b, b_name in enumerate(building_names):
    # Liste des rénovations pour ce bâtiment
    renovations = renovation_names_per_building[b_name]  # exactement R[b] rénovations
    R_b = len(renovations)
    #print(f"Building {b_name} has {R_b} possible renovations.")
    for r in range(R_b):
        r_name = renovations[r]
        for t in range(nbr_annees):
            if X[b, r, t].X > 0.5:  # variable binaire
                results.append({
                    "Building": b_name,
                    "Renovation": r_name,
                    "Year": 2026 + t,
                    "Active": 1
                })

# DataFrame final
df_results = pd.DataFrame(results)
print("Nb lignes export :", len(df_results))



# Pivot : lignes = bâtiment, colonnes = année, valeur = liste des rénovations
df_pivot = df_results.pivot_table(
    index="Building",
    columns="Year",
    values="Renovation",
    aggfunc=lambda x: ', '.join(x)  # concatène les rénovations activées
).fillna("-")  # pour les années sans rénovation

df_pivot.to_excel("solution_optim.xlsx")
print("Fichier Excel généré !")

Budget_per_year = []
Gain_per_year = []
for t in range(nbr_annees):
    budget_t = sum(
        X[b, r, t].X * cost_matrix[b, r]
        for b in range(nbr_batiments)
        for r in range(nbr_renovations_max)
        if cost_matrix[b, r] < float("inf")
    )
    Budget_per_year.append(budget_t)
    gain_t = sum(
        X[b, r, t].X * gains_matrix[b, r]
        for b in range(nbr_batiments)
        for r in range(nbr_renovations_max)
        if gains_matrix[b, r] < float("inf")
    )
    Gain_per_year.append(gain_t)

print("Budget par année :", Budget_per_year)
print("Gain par année   :", Gain_per_year)
budget_df = pd.DataFrame({
    "Year": [2026 + t for t in range(nbr_annees)],
    "Budget_euros": Budget_per_year,
    "Gain_euros": Gain_per_year
})

with pd.ExcelWriter("solution_optim.xlsx", engine="openpyxl", mode="a") as writer:
    budget_df.to_excel(writer, sheet_name="Budget_per_year", index=False)

Nb lignes export : 112
Fichier Excel généré !
Budget par année : [1364671.7172153883, 1364671.7172153883, 1375889.9172153883, 1364671.7172153883, 1364671.7172153883, 47788.63971538806, 47788.63971538806, 79203.63971538807, 47788.63971538806, 79203.63971538807, 47788.63971538806, 47788.63971538806, 57165.439715388056, 47788.63971538806, 47788.63971538806, 47788.63971538806, 47788.63971538806, 79203.63971538807, 47788.63971538806, 57165.439715388056, 47788.63971538806, 47788.63971538806, 47788.63971538806, 79203.63971538807, 47788.63971538806]
